In [ ]:
import os
import numpy as np
import coacd
import trimesh

import shutil
from discoverse import DISCOVERSE_ASSERT_DIR

input_file = "<obj_name>/<obj_name>.obj"
# something like "xxxxx/DISCOVERSE/models/meshes/obj/name/name.obj"

mesh = trimesh.load(input_file, force="mesh")
mesh = coacd.Mesh(mesh.vertices, mesh.faces)
parts = coacd.run_coacd(mesh) # a list of convex hulls.

output_dir = input_file.replace(".obj", "")
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

In [2]:
rgba = "0.5 0.5 0.5 1"

asset_name = input_file.split("/")[-1].replace(".obj", "")

asset_config = f"""
<mujocoinclude>
  <asset>
    <mesh name="{asset_name}" file="obj/{asset_name}/{asset_name}.obj" scale="1 1 1"/>
"""

geom_config = f"""
<mujocoinclude>
  <geom mesh="{asset_name}" rgba="{rgba}" class="obj_visual"/>
"""

for i, part in enumerate(parts):
    output_file = os.path.join(output_dir, f"part_{i}.obj")
    part_mesh = trimesh.Trimesh(vertices=part[0], faces=part[1])
    part_mesh.export(output_file)
    asset_config += '    <mesh name="{}_part_{}" file="obj/{}/{}/part_{}.obj" scale="1 1 1"/>\n'.format(asset_name, i, asset_name, asset_name, i)
    geom_config += '  <geom type="mesh" rgba="{}" mesh="{}_part_{}"'.format(rgba, asset_name, i)
    geom_config += ' class="obj_collision"/>\n'

asset_config += '  </asset>\n</mujocoinclude>'
geom_config += """</mujocoinclude>"""

with open(os.path.join(DISCOVERSE_ASSERT_DIR, f"mjcf/object/{asset_name}_dependencies.xml"), "w") as f:
    f.write(asset_config)

with open(os.path.join(DISCOVERSE_ASSERT_DIR, f"mjcf/object/{asset_name}.xml"), "w") as f:
    f.write(geom_config)